# Hyperparameter Tunning

In [9]:
# EDA

import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y

In [8]:
# Data Load

df_train = pd.read_csv("../dataset/fashion-mnist_train.csv")

X_train , y_train = get_X_y(df_train)

df_test = pd.read_csv("../dataset/fashion-mnist_test.csv")

X_test , y_test = get_X_y(df_test)

X_train

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,5,0,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,1,2,0,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,0,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,0,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
# Data Prep

X_test , X_val , y_test , y_val = train_test_split(
    X_test ,
    y_test ,
    random_state = 42 ,
    stratify = y_test ,
    test_size = 0.5
)

X_train = torch.tensor(
    X_train.values ,
    dtype = torch.float32
)
y_train = torch.tensor(
    y_train.values ,
    dtype = torch.long
)
X_val = torch.tensor(
    X_val.values ,
    dtype = torch.float32
)
y_val = torch.tensor(
    y_val.values ,
    dtype = torch.long
)

train_dataset = TensorDataset(
    X_train ,
    y_train
)
val_dataset = TensorDataset(
    X_val ,
    y_val
)

In [14]:
# Model Builder

def build_model(hidden1 , hidden2 , hidden3 , hidden4 , dropout1 , dropout2 , dropout3 , dropout4) :

    model = nn.Sequential(
    
        nn.Linear(784 , hidden1) ,
        nn.ReLU() ,
        nn.Dropout(dropout1) ,
    
        nn.Linear(hidden1 , hidden2) ,
        nn.ReLU() ,
        nn.Dropout(dropout2) ,
    
        nn.Linear(hidden2 , hidden3) ,
        nn.ReLU() ,
        nn.Dropout(dropout3) ,
    
        nn.Linear(hidden3 , hidden4) ,
        nn.ReLU() ,
        nn.Dropout(dropout4) ,
    
        nn.Linear(hidden4 , 10) 
    )

    return model

In [13]:
# Objective 

def objective(trial) :

    hidden1 = trial.suggest_int(
        "hidden1" ,
        128 ,
        256 ,
        step = 32
    )
    hidden2 = trial.suggest_int(
        "hidden2" ,
        64 ,
        128,
        step = 16
    )
    hidden3 = trial.suggest_int(
        "hidden3" ,
        16 ,
        64 ,
        step = 8
    )
    hidden4 = trial.suggest_int(
        "hidden4" ,
        16 ,
        32 ,
        step = 8
    )
    dropout1 = trial.suggest_float(
        "dropout1" ,
        0.0 ,
        0.5
    )
    dropout2 = trial.suggest_float(
        "dropout2" ,
        0.0 ,
        0.5
    )
    dropout3 = trial.suggest_float(
        "dropout3" ,
        0.0 ,
        0.5
    )
    dropout4 = trial.suggest_float(
        "dropout4" ,
        0.0 ,
        0.5 ,
        step = 0.1
    )
    lr = trial.suggest_float(
        "lr" ,
        1e-5 ,
        1e-2 ,
        log = True
    )
    weight_decay = trial.suggest_float(
        "weight_decay" ,
        1e-6 ,
        1e-2 ,
        log = True
    )
    batch_size = trial.suggest_categorical(
        "batch_size" ,
        [32 , 64 , 128 , 256]
    )

    model = build_model(hidden1 , hidden2 , hidden3 , hidden4 , dropout1 , dropout2 , dropout3 , dropout4)

    optimizer = optim.AdamW(
    model.parameters() ,
    lr = lr ,
    weight_decay = weight_decay
    )

    critrion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        train_dataset ,
        batch_size = batch_size ,
        shuffle = True
    )
    
    val_loader = DataLoader(
        val_dataset ,
        batch_size = batch_size ,
        shuffle = False
    )